In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
 Generate the minimal 7-column results report:
 Fold | System | Rule | p | EER (%) | Rank-1 @ FAR=1% | N_train

 Configuration: seed = 62, p swept 0.00 -> 1.00 in steps of 0.05,
 5 disjoint folds, 80% TRAIN per fold (no test-set evaluation here).
================================================================================

This script REUSES the full evaluation engine from evaluate_pdep_train_5fold.py
(the sweep, the per-fold normalization, EER/Rank-1 computation, etc.) rather
than reimplementing it, so results are guaranteed consistent with that tool.
It only adds a thin final step: reshape the raw sweep into the exact 7 columns
requested and write a single-sheet, seed-tagged .xlsx.

USAGE
-----
    python generate_seed62_report.py
    python generate_seed62_report.py --mat-file scores.mat --seed 62
    python generate_seed62_report.py --seed 62 --p-step 0.05 --output-dir .

Run `python evaluate_pdep_train_5fold.py --help` to see every other knob
(rules, systems, FAR target, normalization, etc.) -- they all pass through
via the Config object below if you need to change them for this report too.
================================================================================
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

# ------------------------------------------------------------------------
# Reuse the evaluation engine instead of reimplementing it.
# Assumes evaluate_pdep_train_5fold.py is on the Python path (same folder,
# or installed). Adjust sys.path.insert(...) below if it lives elsewhere.
# ------------------------------------------------------------------------
#sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.path.insert(0, str(Path.cwd()))

from evaluate_pdep_train_5fold import (  # noqa: E402
    Config,
    DataConfig,
    SplitConfig,
    SweepConfig,
    MetricConfig,
    PDependentTrainEvaluator,
)


# ==============================================================================
#  CONFIGURATION FOR THIS SPECIFIC REPORT
#  Everything specific to "the seed=62 report" is isolated here.
# ==============================================================================

DEFAULT_MAT_FILE = "870d37a7-673d-49ff-8a98-47b9b519237f.mat"
SEED = 42
P_STEP = 0.05
N_FOLDS = 5
OUTPUT_DIR = "./Results"
# The exact columns and their order in the final report, and how each maps
# from the evaluator's raw sweep output (Train_EER, Train_Rank1, ... are
# fractions in [0, 1]; here they are rescaled to percentages for display).
REPORT_COLUMNS = {
    "Fold":              lambda df: df["Fold"],
    "System":            lambda df: df["System"],
    "Rule":              lambda df: df["Rule"],
    "p":                 lambda df: df["p"],
    "EER (%)":           lambda df: (df["Train_EER"] * 100).round(4),
    "Rank-1 @ FAR=1%":   lambda df: (df["Train_Rank1"] * 100).round(4),
    "N_train":           lambda df: df["N_Train_Subjects"],
}


# ==============================================================================
#  STEP 1 -- RUN THE 5-FOLD SWEEP (delegated to the reusable evaluator)
# ==============================================================================

def run_sweep(mat_file: str, seed: int, p_step: float, n_folds: int) -> pd.DataFrame:
    """
    Execute the full (fold x system x rule x p) sweep on the 80% TRAIN folds.

    Returns the raw, long-format results as a DataFrame with one row per
    (Fold, System, Rule, p) combination.
    """
    config = Config(
        data=DataConfig(mat_file=mat_file),
        split=SplitConfig(n_folds=n_folds, seed=seed, shuffle=True),
        sweep=SweepConfig(p_min=0.0, p_max=1.0, p_step=p_step),
        metrics=MetricConfig(far_target=0.01, confidence_level=0.95),
    )

    evaluator = PDependentTrainEvaluator(config)
    evaluator.load_data()
    raw_results = evaluator.run()   # -> DataFrame, one row per evaluation
    return raw_results


# ==============================================================================
#  STEP 2 -- RESHAPE TO THE REQUESTED 7 COLUMNS
# ==============================================================================

def build_minimal_report(raw_results: pd.DataFrame) -> pd.DataFrame:
    """Select and rename exactly the columns requested, in the requested order."""
    report = pd.DataFrame({
        column_name: builder(raw_results)
        for column_name, builder in REPORT_COLUMNS.items()
    })
    return report.sort_values(
        ["System", "Rule", "p", "Fold"]
    ).reset_index(drop=True)


# ==============================================================================
#  STEP 3 -- WRITE THE FORMATTED, SEED-TAGGED XLSX
# ==============================================================================

def write_report(report: pd.DataFrame, seed: int, p_step: float,
                 n_folds: int, output_dir: Path) -> Path:
    """Write `report` to a single-sheet workbook whose filename includes the seed."""
    font_name = "Arial"
    theme_color = "0B5394"

    header_fill = PatternFill("solid", fgColor=theme_color)
    header_font = Font(name=font_name, bold=True, color="FFFFFF", size=10)
    title_font = Font(name=font_name, bold=True, size=13, color=theme_color)
    subtitle_font = Font(name=font_name, italic=True, size=9, color="555555")
    cell_font = Font(name=font_name, size=10)
    border = Border(*(Side(style="thin", color="D9D9D9") for _ in range(4)))

    workbook = Workbook()
    worksheet = workbook.active
    worksheet.title = "Results"

    n_systems = report["System"].nunique()
    n_rules = report["Rule"].nunique()
    n_p_values = report["p"].nunique()

    worksheet.cell(row=1, column=1,
                   value=f"P-Dependent Fusion Rules \u2014 5-Fold CV on 80% TRAIN "
                         f"(seed={seed})").font = title_font
    worksheet.cell(row=2, column=1,
                   value=f"{n_systems} systems x {n_rules} rules x {n_p_values} p-values "
                         f"(0.00 to 1.00, step {p_step}) x {n_folds} disjoint folds "
                         f"= {len(report)} rows. Each fold's TRAIN subset (~80%) is "
                         "normalized independently (min-max fit on that fold's train "
                         "subjects only).").font = subtitle_font

    header_row = 4
    for col_idx, column in enumerate(report.columns, start=1):
        cell = worksheet.cell(row=header_row, column=col_idx, value=column)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border

    for row_offset, record in enumerate(report.itertuples(index=False), start=1):
        for col_idx, value in enumerate(record, start=1):
            column = report.columns[col_idx - 1]
            cell = worksheet.cell(row=header_row + row_offset, column=col_idx, value=value)
            cell.font = cell_font
            cell.border = border
            if column == "p":
                cell.number_format = "0.00"
            elif column in ("EER (%)", "Rank-1 @ FAR=1%"):
                cell.number_format = "0.0000"

    worksheet.freeze_panes = worksheet.cell(row=header_row + 1, column=1)
    _autofit_columns(worksheet, len(report.columns), header_row, len(report))

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"PDependent_5FoldCV_seed{seed}.xlsx"
    workbook.save(output_path)
    return output_path


def _autofit_columns(worksheet, n_columns: int, header_row: int,
                     n_data_rows: int, minimum: int = 10, maximum: int = 28) -> None:
    for col_idx in range(1, n_columns + 1):
        letter = get_column_letter(col_idx)
        longest = max(
            (len(str(worksheet.cell(row=r, column=col_idx).value))
             for r in range(header_row, header_row + n_data_rows + 1)
             if worksheet.cell(row=r, column=col_idx).value is not None),
            default=minimum,
        )
        worksheet.column_dimensions[letter].width = min(max(longest + 2, minimum), maximum)


# ==============================================================================
#  COMMAND-LINE ENTRY POINT
# ==============================================================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mat-file", default=DEFAULT_MAT_FILE,
                        help="Path to the .mat file with the score matrices.")
    parser.add_argument("--seed", type=int, default=SEED)
    parser.add_argument("--p-step", type=float, default=P_STEP)
    parser.add_argument("--n-folds", type=int, default=N_FOLDS)
    parser.add_argument("--output-dir", default=".",
                        help="Directory the .xlsx report is written into.")
    return parser.parse_args()


if __name__ == "__main__":

    print(f"Running 5-fold sweep: seed={SEED}, p_step={P_STEP}, "
          f"n_folds={N_FOLDS} ...")
    raw_results = run_sweep(DEFAULT_MAT_FILE, SEED, P_STEP, N_FOLDS)
    print(f"Sweep complete: {len(raw_results)} rows.")

    report = build_minimal_report(raw_results)
    output_path = write_report(report, SEED, P_STEP, N_FOLDS, Path(OUTPUT_DIR))

    print(f"Report written to: {output_path}")
    print(report.head(10).to_string(index=False))
    




Running 5-fold sweep: seed=42, p_step=0.05, n_folds=5 ...
Sweep complete: 8400 rows.
Report written to: Results\PDependent_5FoldCV_seed42.xlsx
 Fold System       Rule    p  EER (%)  Rank-1 @ FAR=1%  N_train
    1     S1 Luhandjula 0.00   4.3492          87.8935      413
    2     S1 Luhandjula 0.00   4.1144          87.6214      413
    3     S1 Luhandjula 0.00   3.8741          89.3462      414
    4     S1 Luhandjula 0.00   4.3581          86.4407      414
    5     S1 Luhandjula 0.00   4.6008          87.8935      414
    1     S1 Luhandjula 0.05   4.1136          88.1356      413
    2     S1 Luhandjula 0.05   3.8835          87.6214      413
    3     S1 Luhandjula 0.05   3.8738          89.3462      414
    4     S1 Luhandjula 0.05   4.3584          86.4407      414
    5     S1 Luhandjula 0.05   4.3584          87.8935      414
